In [1]:
import base64

from google.genai import types

_BYTES_KEY = "__bytes__"


def _encode_bytes(obj):
    """Recursively replace bytes (e.g. thought_signature) with a base64 marker."""
    if isinstance(obj, bytes):
        return {_BYTES_KEY: base64.b64encode(obj).decode("ascii")}
    if isinstance(obj, dict):
        return {k: _encode_bytes(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_encode_bytes(v) for v in obj]
    return obj


def serialize_conversation(conversation):
    """Faithful, JSON-safe view of a mixed list (plain dicts + genai Content)."""
    out = []
    for item in conversation:
        if isinstance(item, types.Content):
            out.append(_encode_bytes(item.model_dump(exclude_none=True)))
        else:
            out.append(_encode_bytes(item))
    return out

In [3]:
import json
from pathlib import Path

def test_serialize_conversation_on_file(path_str="/Users/clement/Desktop/sunraise/src/conversation/conversation_2026-06-26/conversation_google_2026-06-26_09-40-57_047626.json"):
    path = Path(path_str)
    if not path.exists():
        path = Path.cwd() / path

    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)

    if isinstance(raw, dict):
        payload = raw.get("conversation") or raw.get("messages") or raw.get("items") or [raw]
    else:
        payload = raw

    serialized = serialize_conversation(payload)

    assert isinstance(serialized, list), f"Expected a list, got {type(serialized).__name__}"
    assert len(serialized) == len(payload), "Serialized conversation length changed"

    json.dumps(serialized, ensure_ascii=False)

    print(f"Loaded: {path}")
    print(f"Serialized {len(serialized)} item(s)")
    return serialized

test_serialize_conversation_on_file()

Loaded: /Users/clement/Desktop/sunraise/src/conversation/conversation_2026-06-26/conversation_google_2026-06-26_09-40-57_047626.json
Serialized 3 item(s)


[{'role': 'user', 'parts': [{'text': 'bonjour'}]},
 {'role': 'model',
  'parts': [{'text': "Bonjour ! Comment puis-je vous aider aujourd'hui ?"}]},
 {'role': 'user', 'parts': [{'text': '/q'}]}]